In [40]:
import pandas as pd

# PREPROCESSING

In [41]:
df = pd.read_csv('merged_file.csv')

## check duplicates

In [42]:
df.duplicated().sum()

np.int64(0)

In [43]:
df = df.drop_duplicates(keep='first').reset_index(drop=True)
df.shape

(46628, 12)

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46628 entries, 0 to 46627
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               46628 non-null  object
 1   brand            46628 non-null  object
 2   price            46628 non-null  object
 3   start_time       46628 non-null  object
 4   start_day        46628 non-null  object
 5   end_day          46628 non-null  object
 6   end_time         46628 non-null  object
 7   trip_time        46628 non-null  object
 8   destination      46628 non-null  object
 9   hand_luggage     37572 non-null  object
 10  checked_baggage  37572 non-null  object
 11  crawl_date       46628 non-null  object
dtypes: object(12)
memory usage: 4.3+ MB


## Take_place, Destination

In [45]:
df['destination'].unique()

array(['Phú Quốc (PQC)\nSân bay Phú Quốc',
       'Đà Nẵng (DAD)\nSân bay Đà Nẵng',
       'Hà Nội (HAN)\r\nSân bay Nội Bài',
       'Hà Nội (HAN)\r\nSân bay Nội Bài\r\nNhà ga 1',
       'Hải Phòng (HPH)\nSân bay quốc tế Cát Bi',
       'Đà Lạt (DLI)\r\nSân bay Liên Khương',
       'Nha Trang (CXR)\nSân bay Cam Ranh'], dtype=object)

In [46]:
df['destination'] = df['destination'].replace(
    'Hà Nội (HAN)\r\nSân bay Nội Bài\r\nNhà ga 1',
    'Hà Nội (HAN)\r\nSân bay Nội Bài'
)
df['destination'].value_counts()

destination
Hà Nội (HAN)\r\nSân bay Nội Bài            19302
Đà Nẵng (DAD)\nSân bay Đà Nẵng             12839
Phú Quốc (PQC)\nSân bay Phú Quốc            5288
Hải Phòng (HPH)\nSân bay quốc tế Cát Bi     4186
Nha Trang (CXR)\nSân bay Cam Ranh           3178
Đà Lạt (DLI)\r\nSân bay Liên Khương         1835
Name: count, dtype: int64

## brand

In [47]:
df['brand'].nunique(), df['brand'].unique()

(4,
 array(['Bamboo Airways', 'Vietravel Airlines', 'Vietnam Airlines',
        'VietJet Air'], dtype=object))

## price

In [48]:
df['price'].head()

0    1.245.267 VND/khách
1    1.224.930 VND/khách
2    1.535.463 VND/khách
3    1.567.000 VND/khách
4    1.567.000 VND/khách
Name: price, dtype: object

In [49]:
# Clean and convert the price column
df['price'] = df['price'].str.extract(r'([\d\.]+)')
df['price'] = df['price'].str.replace('.', '', regex=False).astype(int)
df['price'].head()

0    1245267
1    1224930
2    1535463
3    1567000
4    1567000
Name: price, dtype: int64

## time

In [50]:
df['start_time'] = df['start_time'].str.replace('h', ':')
df['end_time'] = df['end_time'].str.replace('h', ':')

In [51]:
df['start_hour'] = df['start_time'].str.split(':').str[0].astype(int)
df['end_hour'] = df['end_time'].str.split(':').str[0].astype(int)
df[['start_time', 'start_hour', 'end_time', 'end_hour']].head()

,start_time,start_hour,end_time,end_hour
0,16:30,16,17:30,17
1,17:25,17,18:25,18
2,17:25,17,18:25,18
3,15:40,15,16:40,16
4,17:10,17,18:15,18


In [52]:
df['start_hour'] = pd.cut(
    df['start_hour'],
    bins=[0, 3, 9, 15, 21, 24],
    labels=['EarlyMorning', 'Morning', 'Afternoon', 'Evening', 'LateNight'],
    include_lowest=True
)
df['end_hour'] = pd.cut(
    df['end_hour'],
    bins=[0, 3, 9, 15, 21, 24],
    labels=['EarlyMorning', 'Morning', 'Afternoon', 'Evening', 'LateNight'],
    include_lowest=True
)
df[['start_time', 'start_hour', 'end_time', 'end_hour']].head()

,start_time,start_hour,end_time,end_hour
0,16:30,Evening,17:30,Evening
1,17:25,Evening,18:25,Evening
2,17:25,Evening,18:25,Evening
3,15:40,Afternoon,16:40,Evening
4,17:10,Evening,18:15,Evening


In [53]:
time_parts = df['trip_time'].str.extract(r'(?:(?P<hour>\d+)h)?\s*(?:(?P<minute>\d+)m)?')
time_parts = time_parts.astype(float).fillna(0)

df['trip_mins'] = time_parts['hour'] * 60 + time_parts['minute']

df[['trip_time', 'trip_mins']].value_counts()

trip_time  trip_mins
2h 10m     130.0        11294
2h 5m      125.0         8412
1h 25m     85.0          6596
1h 0m      60.0          4963
1h 20m     80.0          4882
2h 0m      120.0         3342
1h 5m      65.0          2608
55m        55.0          2082
1h 15m     75.0           906
1h 30m     90.0           761
1h 55m     115.0          413
45m        45.0           334
2h 15m     135.0           26
1h 10m     70.0             4
50m        50.0             4
2h 25m     145.0            1
Name: count, dtype: int64

In [54]:
df['trip_mins'] = df['trip_mins'].astype(int)
df.drop(columns=['start_time', 'end_time', 'trip_time'], inplace=True)

## day

In [55]:
df['start_day'].unique(), df['start_day'].nunique(), df['end_day'].unique(), df['end_day'].nunique()

(array(['02 thg 5', '03 thg 5', '04 thg 5', '05 thg 5', '06 thg 5',
        '07 thg 5', '08 thg 5', '09 thg 5', '10 thg 5', '11 thg 5',
        '21 thg 4', '22 thg 4', '23 thg 4', '24 thg 4', '25 thg 4',
        '26 thg 4', '27 thg 4', '01 thg 5', '28 thg 4', '29 thg 4',
        '30 thg 4'], dtype=object),
 21,
 array(['02 thg 5', '03 thg 5', '04 thg 5', '05 thg 5', '06 thg 5',
        '07 thg 5', '08 thg 5', '09 thg 5', '10 thg 5', '11 thg 5',
        '21 thg 4', '22 thg 4', '23 thg 4', '24 thg 4', '25 thg 4',
        '26 thg 4', '27 thg 4', '01 thg 5', '28 thg 4', '29 thg 4',
        '30 thg 4', '12 thg 5'], dtype=object),
 22)

In [56]:
def convert_vn_date(date_str, year=2025):
    day, month = date_str.strip().split(' thg ')
    dt = pd.to_datetime(f"{day}-{int(month):02d}-{year}", dayfirst=True)
    return dt

df['start_day'] = df['start_day'].apply(lambda x: convert_vn_date(x, 2025))
df['end_day'] = df['end_day'].apply(lambda x: convert_vn_date(x, 2025))
df[['start_day', 'end_day']].head(), df['start_day'].dtype, df['end_day'].dtype

(   start_day    end_day
 0 2025-05-02 2025-05-02
 1 2025-05-02 2025-05-02
 2 2025-05-02 2025-05-02
 3 2025-05-02 2025-05-02
 4 2025-05-02 2025-05-02,
 dtype('<M8[ns]'),
 dtype('<M8[ns]'))

In [57]:
holidays = [
    pd.Timestamp('2025-04-30').date(),
    pd.Timestamp('2025-05-01').date(),
]

nearby_holidays = [
    pd.Timestamp('2025-04-29').date(),
    pd.Timestamp('2025-05-02').date(),
    pd.Timestamp('2025-05-03').date(),
    pd.Timestamp('2025-05-04').date(),
]

def is_holiday(date):
    d = date.date()
    if d in holidays:
        return 3
    elif d in nearby_holidays:
        return 2
    elif d.weekday() >= 4:  # Friday = 4
        return 1
    else:
        return 0
    
df['is_holiday'] = df['start_day'].apply(is_holiday)
df[['start_day', 'is_holiday']].value_counts().head(5)

start_day   is_holiday
2025-05-05  0             2543
2025-04-27  1             2530
2025-05-02  2             2529
2025-05-03  2             2470
2025-05-04  2             2456
Name: count, dtype: int64

In [58]:
df['start_day'].dtype

dtype('<M8[ns]')

In [59]:
df['crawl_date'].nunique(), df['crawl_date'].unique()

(34,
 array(['01-05-2025', '02-05-2025', '03-05-2025', '04-05-2025',
        '05-05-2025', '06-05-2025', '07-04-2025', '07-05-2025',
        '08-05-2025', '09-04-2025', '09-05-2025', '10-04-2025',
        '10-05-2025', '11-04-2025', '12-04-2025', '13-04-2025',
        '14-04-2025', '15-04-2025', '16-04-2025', '17-04-2025',
        '18-04-2025', '19-04-2025', '20-04-2025', '21-04-2025',
        '22-04-2025', '23-04-2025', '24-04-2025', '25-04-2025',
        '26-04-2025', '27-04-2025', '28-04-2025', '29-04-2025',
        '30-04-2025', '08-04-2025'], dtype=object))

In [60]:
df['crawl_date'] = pd.to_datetime(df['crawl_date'], format='%d-%m-%Y', errors='coerce')
df['crawl_date'].nunique(), df['crawl_date'].unique()

(34,
 <DatetimeArray>
 ['2025-05-01 00:00:00', '2025-05-02 00:00:00', '2025-05-03 00:00:00',
  '2025-05-04 00:00:00', '2025-05-05 00:00:00', '2025-05-06 00:00:00',
  '2025-04-07 00:00:00', '2025-05-07 00:00:00', '2025-05-08 00:00:00',
  '2025-04-09 00:00:00', '2025-05-09 00:00:00', '2025-04-10 00:00:00',
  '2025-05-10 00:00:00', '2025-04-11 00:00:00', '2025-04-12 00:00:00',
  '2025-04-13 00:00:00', '2025-04-14 00:00:00', '2025-04-15 00:00:00',
  '2025-04-16 00:00:00', '2025-04-17 00:00:00', '2025-04-18 00:00:00',
  '2025-04-19 00:00:00', '2025-04-20 00:00:00', '2025-04-21 00:00:00',
  '2025-04-22 00:00:00', '2025-04-23 00:00:00', '2025-04-24 00:00:00',
  '2025-04-25 00:00:00', '2025-04-26 00:00:00', '2025-04-27 00:00:00',
  '2025-04-28 00:00:00', '2025-04-29 00:00:00', '2025-04-30 00:00:00',
  '2025-04-08 00:00:00']
 Length: 34, dtype: datetime64[ns])

In [61]:
df['days_left'] = (pd.to_datetime(df['start_day']) - pd.to_datetime(df['crawl_date'])).dt.days
df[['start_day', 'crawl_date', 'days_left']].value_counts().head()

start_day   crawl_date  days_left
2025-05-04  2025-04-22  12           137
2025-04-28  2025-04-22  6            137
            2025-04-23  5            137
2025-05-05  2025-04-22  13           136
2025-04-28  2025-04-17  11           136
Name: count, dtype: int64

In [62]:
df['days_left'].value_counts()

days_left
11    2537
10    2532
8     2510
9     2501
12    2499
14    2491
13    2443
5     2406
7     2396
6     2388
4     2379
15    2355
3     2282
2     2251
16    2158
1     2040
17    2030
18    1955
19    1717
20    1627
22     103
24     103
23     102
25     100
21      98
28      97
29      94
30      94
26      92
27      92
0       79
31      41
32      12
34      12
33      12
Name: count, dtype: int64

In [63]:
df = df[~((df['days_left'] == 0) | (df['days_left'] > 20))]

In [64]:
df.drop(columns=['start_day', 'end_day', 'crawl_date'], inplace=True)

## luggage:

In [65]:
df[['hand_luggage', 'checked_baggage']].value_counts()

hand_luggage                checked_baggage  
Hành lý xách tay 7 kg       Hành lý 0 kg         19817
Hành lý xách tay 1 x 12 kg  Hành lý 23 kg         8890
                            Hành lý 1 x 23 kg     6596
Hành lý xách tay 7 kg       Hành lý 20 kg          464
Hành lý xách tay 10 kg      Hành lý 23 kg          458
                            Hành lý 1 x 23 kg      437
Hành lý xách tay 1 x 12 kg  Hành lý 0 kg           387
Hành lý xách tay 10 kg      Hành lý 0 kg            12
Name: count, dtype: int64

In [66]:
import re

def format_lugggage(string):
    if pd.isna(string):
        return None
    number=re.findall(r'\d+',str(string))
    if len(number)>=2: # ve co hanh ly xach tay format theo kien hang 1x12kg
        return int(number[0])*int(number[1])  
    elif len(number)==1:
        return int(number[0])
    else:
        return None

column=['checked_baggage','hand_luggage']
for c in column:
    df[c]=df[c].apply(format_lugggage)

# for c in column:
#     mode_val = df[c].mode()[0]  # Lấy mode (giá trị phổ biến nhất)
#     df[c].fillna(mode_val) 

In [67]:
import numpy as np

In [68]:
df['hand_luggage'] = df.groupby('id')['hand_luggage']\
    .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))

df['checked_baggage'] = df.groupby('id')['checked_baggage']\
    .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))

In [69]:
df[['hand_luggage', 'checked_baggage']].isnull().sum()

hand_luggage       165
checked_baggage    165
dtype: int64

In [70]:
df[df['hand_luggage'].isnull() | df['checked_baggage'].isnull()]

,id,brand,price,destination,hand_luggage,checked_baggage,start_hour,end_hour,trip_mins,is_holiday,days_left
616,PQC0171,Vietnam Airlines,923544,Phú Quốc (PQC)\nSân bay Phú Quốc,NaN,NaN,Morning,Morning,55,0,14
628,PQC0172,Vietnam Airlines,811725,Phú Quốc (PQC)\nSân bay Phú Quốc,NaN,NaN,Morning,Morning,55,0,15
643,PQC0121,VietJet Air,837000,Phú Quốc (PQC)\nSân bay Phú Quốc,NaN,NaN,Afternoon,Evening,60,0,16
806,PQC0171,Vietnam Airlines,879056,Phú Quốc (PQC)\nSân bay Phú Quốc,NaN,NaN,Morning,Morning,55,0,12
821,PQC0172,Vietnam Airlines,879056,Phú Quốc (PQC)\nSân bay Phú Quốc,NaN,NaN,Morning,Morning,55,0,13
...,...,...,...,...,...,...,...,...,...,...,...
43455,CXR0002,VietJet Air,818157,Nha Trang (CXR)\nSân bay Cam Ranh,NaN,NaN,Afternoon,Afternoon,60,0,20
43675,CXR0015,VietJet Air,1286535,Nha Trang (CXR)\nSân bay Cam Ranh,NaN,NaN,Afternoon,Afternoon,60,0,16
43676,CXR0015,VietJet Air,1142489,Nha Trang (CXR)\nSân bay Cam Ranh,NaN,NaN,Afternoon,Afternoon,60,0,15
43874,CXR0035,VietJet Air,831915,Nha Trang (CXR)\nSân bay Cam Ranh,NaN,NaN,Evening,Evening,60,1,16


In [71]:
# VietJet Air
mask = (df['brand'] == 'VietJet Air') & (df['hand_luggage'].isna())
df.loc[mask, 'hand_luggage'] = 7
df.loc[mask, 'checked_baggage'] = 0

# Vietnam Airlines
mask = (df['brand'] == 'Vietnam Airlines') & (df['hand_luggage'].isna())
df.loc[mask, 'hand_luggage'] = 12
df.loc[mask, 'checked_baggage'] = 23

In [72]:
df[['hand_luggage', 'checked_baggage', 'brand']].isnull().sum()

hand_luggage       0
checked_baggage    0
brand              0
dtype: int64

In [73]:
df['hand_luggage'] = df['hand_luggage'].astype(int)
df['checked_baggage'] = df['checked_baggage'].astype(int)

In [74]:
df.head()

,id,brand,price,destination,hand_luggage,checked_baggage,start_hour,end_hour,trip_mins,is_holiday,days_left
0,PQC0002,Bamboo Airways,1245267,Phú Quốc (PQC)\nSân bay Phú Quốc,7,0,Evening,Evening,60,2,1
1,PQC0323,Vietravel Airlines,1224930,Phú Quốc (PQC)\nSân bay Phú Quốc,7,0,Evening,Evening,60,2,1
2,PQC0292,Vietnam Airlines,1535463,Phú Quốc (PQC)\nSân bay Phú Quốc,12,23,Evening,Evening,60,2,1
3,PQC0109,VietJet Air,1567000,Phú Quốc (PQC)\nSân bay Phú Quốc,7,0,Afternoon,Evening,60,2,1
4,PQC0132,VietJet Air,1567000,Phú Quốc (PQC)\nSân bay Phú Quốc,7,0,Evening,Evening,65,2,1


In [76]:
df.to_csv('cleaned_file.csv', index=False)